# VitaNexus-RX — Full HGNN Training
Run only after the LightGBM notebook completes. In **Runtime → Change runtime type**, select runtime version **2026.07** (Python 3.12) and a **T4 GPU** (or L4/A100 if available). The current default Python 3.13 Colab image is not accepted because its scientific-package overlay can contain inconsistent NumPy binaries. The pipeline uses CPU only as an explicit fallback and prints the actual device.

## 1. Configuration — edit only this cell if needed

In [ ]:
import sys
if sys.version_info[:2] != (3, 12):
    raise RuntimeError('Select Runtime -> Change runtime type -> Runtime Version 2026.07 (Python 3.12), choose a GPU, reconnect, then Run all.')
DRIVE_ROOT = '/content/drive/MyDrive/VitaNexus-RX-ML'
REPOSITORY_URL = 'https://github.com/Sravanramaraju/VitaNexus-RX.git'
BRANCH = 'codex/faers-ml-clinical-integration'
REPOSITORY = '/content/VitaNexus-RX'
LOCAL_WORK = '/content/vitanexus-ml-work'

## 2. Mount Google Drive — authorization appears here

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Acquire the exact committed repository branch

In [ ]:
import pathlib, subprocess
repo = pathlib.Path(REPOSITORY)
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPOSITORY_URL, REPOSITORY], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=repo, check=True)
    if subprocess.check_output(['git', 'status', '--porcelain'], cwd=repo, text=True).strip():
        raise RuntimeError('Existing Colab checkout is dirty; use a fresh runtime instead of discarding files.')
    subprocess.run(['git', 'checkout', BRANCH], cwd=repo, check=True)
    subprocess.run(['git', 'merge', '--ff-only', f'origin/{BRANCH}'], cwd=repo, check=True)

## 4. Install Colab dependencies without replacing Colab's CUDA PyTorch

In [ ]:
import importlib.metadata, os, signal, subprocess, sys, time
from pathlib import Path

requirements = Path(REPOSITORY) / 'ml' / 'requirements-colab.txt'
pins = dict(line.split('==', 1) for line in requirements.read_text().splitlines() if '==' in line and not line.lstrip().startswith('#'))
installed = {}
for package in pins:
    try:
        installed[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed[package] = None
changes = {package: {'installed': installed[package], 'required': required} for package, required in pins.items() if installed[package] != required}
probe_code = 'import numpy; from numpy._core.umath import _slice; import pandas, pyarrow, scipy, sklearn, lightgbm'
probe = subprocess.run([sys.executable, '-c', probe_code], capture_output=True, text=True)
broken_stack = probe.returncode != 0
if changes or broken_stack:
    if changes:
        print(f'Installing constrained dependencies: {changes}', flush=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)], check=True)
    if broken_stack:
        print(f'Scientific-stack import probe failed; repairing binary packages:\n{probe.stderr}', flush=True)
        repair = [f'{name}=={pins[name]}' for name in ('numpy', 'pandas', 'pyarrow', 'scipy', 'scikit-learn', 'lightgbm')]
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-cache-dir', '--no-deps', *repair], check=True)
    print('DEPENDENCIES INSTALLED OR REPAIRED. Colab is restarting to prevent mixed NumPy/SciPy modules. After reconnection, choose Runtime -> Run all once more.', flush=True)
    time.sleep(2)
    os.kill(os.getpid(), signal.SIGKILL)
else:
    print('Constrained dependencies and binary import probe passed; no runtime restart required.')

## 5. Resolve paths and stage verified immutable data

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, f'{REPOSITORY}/ml/colab')
from colab_common import ColabPaths, configure, stage_dataset
paths = ColabPaths(Path(DRIVE_ROOT), Path(REPOSITORY), Path(LOCAL_WORK))
configure(paths)
stage_dataset(paths)

## 6. Strict HGNN preflight: CUDA, full LightGBM dependency and temporal guards

In [ ]:
from colab_common import hgnn_preflight
preflight = hgnn_preflight(paths, BRANCH)
if not preflight['runtime']['cudaAvailable']:
    print('WARNING: CUDA is unavailable. Stop and select a GPU runtime unless CPU fallback is intentional.')
else:
    print('CUDA ACTIVE:', preflight['runtime']['gpu'], preflight['runtime']['gpuMemoryGB'], 'GB')

## 7. Train/select, final-refit, calibrate and evaluate the corrected full HGNN

In [ ]:
import os, subprocess, sys
subprocess.run([sys.executable, '-m', 'vitanexus_ml.cli', 'train-hgnn'], cwd=REPOSITORY, env=os.environ.copy(), check=True)

## 8. Verify full HGNN promotion and holdout report

In [ ]:
import json
manifest = json.loads((paths.drive_models / 'hgnn_training_manifest.json').read_text())
assert manifest['fastMode'] is False and manifest['fullFinalData'] is True
assert manifest['selectionTrainWindow'] == ['2022Q1', '2024Q4']
assert manifest['validationWindow'] == ['2025Q1', '2025Q2']
metrics = json.loads((paths.drive_reports / 'hgnn_metrics.json').read_text())
assert 'holdout2026' in metrics
print('HGNN FULL TRAINING COMPLETE:', json.dumps(metrics['holdout2026'], indent=2))

## 9. Export combined verified LightGBM + HGNN inference bundle

In [ ]:
from vitanexus_ml.artifact_bundle import export_inference_bundle, verify_inference_bundle
output = paths.drive_exports / 'vitanexus_full_inference'
if (output / 'inference_bundle_manifest.json').exists():
    result = verify_inference_bundle(output, require_all=True)
else:
    result = export_inference_bundle(paths.drive_models, paths.drive_reports, output, component='all')
print(json.dumps(result, indent=2))

## 10. Final summary
Download or Drive-sync `exports/vitanexus_full_inference` and use the repository's validated local import command. If Colab disconnects, reconnect and Run all: completed Drive checkpoints are detected, `latest` resumes the next epoch, and `best` remains separate.